# Stage 13: emission uncertainty gate

CPU-only validation notebook. It reuses Stage 12B/12C OOF artifacts and tests conservative correction caps, risk shrinkage, and small cross-family blends. It trains no neural model and creates no submission.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir = drive_root / 'artifacts'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)


## Reuse Stage 12 artifacts

This inexpensive stage expects the completed Stage 12B and Stage 12C Drive artifacts. If they are missing, run notebooks 280 and 290 first because rebuilding sixteen GPU folds from a CPU notebook is not useful.


In [ ]:
STAGE12B_RUN_ID = 'stage12b_learned_emission_tcn_full_v001'
STAGE12C_RUN_ID = 'stage12c_spatial_kbest_lattice_full_v001'
stage12b_run = artifact_dir / STAGE12B_RUN_ID
stage12c_run = artifact_dir / STAGE12C_RUN_ID
assert (stage12b_run / 'oof_emission.parquet').is_file(), 'Run notebook 280 first'
assert (stage12c_run / 'gate_summary.json').is_file(), 'Run notebook 290 first'
assert json.loads((stage12b_run / 'gate_summary.json').read_text())['promoted_to_spatial_emission_audit'] is True
assert json.loads((stage12c_run / 'gate_summary.json').read_text())['promoted_to_all_train_alignment'] is False
print('Stage 12 artifacts verified')


## Run nested uncertainty gate

Use `LIMIT_WELLS = None`. This should finish on CPU much faster than Stage 12C.


In [ ]:
RUN_ID = 'stage13_emission_uncertainty_gate_full_v001'
LIMIT_WELLS = None
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'gate_summary.json').is_file():
    command = [
        'uv', 'run', 'rogii-emission-uncertainty',
        '--config', 'configs/experiment/stage13_emission_uncertainty_gate.yaml',
        '--stage12b-run', str(stage12b_run), '--stage12c-run', str(stage12c_run),
        '--artifact-dir', str(artifact_dir), '--run-id', RUN_ID,
    ]
    if LIMIT_WELLS is not None:
        command += ['--limit-wells', str(LIMIT_WELLS)]
    subprocess.run(command, cwd=repo_dir, check=True)
else:
    print('Reusing completed run:', run_dir)


## Decision

Standard candidates may include cross-family blends. Spatial and typewell promotion uses only their own target-hidden correction magnitude and roughness.


In [ ]:
summary = json.loads((run_dir / 'gate_summary.json').read_text(encoding='utf-8'))
families = summary['family_reports']
decision = {
    'promoted_to_all_train_uncertainty_model': summary['promoted_to_all_train_uncertainty_model'],
    'standard_base_rmse': families['fold']['base_metrics']['pooled_rmse'],
    'standard_nested_rmse': families['fold']['nested_metrics']['pooled_rmse'],
    'standard_delta': families['fold']['nested_rmse_delta'],
    'spatial_base_rmse': families['spatial_fold']['base_metrics']['pooled_rmse'],
    'spatial_nested_rmse': families['spatial_fold']['nested_metrics']['pooled_rmse'],
    'spatial_delta': families['spatial_fold']['nested_rmse_delta'],
    'typewell_base_rmse': families['typewell_fold']['base_metrics']['pooled_rmse'],
    'typewell_nested_rmse': families['typewell_fold']['nested_metrics']['pooled_rmse'],
    'typewell_delta': families['typewell_fold']['nested_rmse_delta'],
    'bootstrap_95pct': {name: [report['bootstrap']['ci_2_5'], report['bootstrap']['ci_97_5']] for name, report in families.items()},
    'error_correlations': summary['error_correlations'],
    'robust_local_profile': summary['robust_local_profile'],
    'gates': summary['gates'], 'next_step': summary['next_step'],
}
decision


In [ ]:
import pandas as pd
profile_rows = []
for family, report in families.items():
    for name, values in report['profile_reports'].items():
        profile_rows.append({
            'family': family, 'profile': name, 'rmse_delta': values['pooled_rmse_delta'],
            'worst_fold_delta': max(values['fold_deltas'].values()),
            'p90_delta': values['candidate_metrics']['well_rmse_p90'] - values['base_metrics']['well_rmse_p90'],
            'worst10_delta': values['candidate_metrics']['worst_10pct_sse_share'] - values['base_metrics']['worst_10pct_sse_share'],
        })
pd.DataFrame(profile_rows).sort_values(['family', 'rmse_delta']).reset_index(drop=True)


## Send back

Send both the decision dictionary and profile table. No Kaggle submission is produced.
